In [6]:
hdi = pd.read_csv("../data/raw/HDI_countries_long.csv")
happiness = pd.read_csv("../data/raw/world_happiness_report_2011_2025.csv")
iso = pd.read_csv("../data/raw/country-iso-alpha-2-6.csv")
gdi = pd.read_excel("../data/raw/hdr-data-GDI.xlsx", sheet_name="Data")

In [7]:
print(hdi.shape)
print(happiness.shape)
print(iso.shape)
print(gdi.shape)

(6630, 9)
(2116, 13)
(191, 2)
(66537, 12)


In [4]:
print(hdi["year"].min(), hdi["year"].max())
print(happiness["year"].min(), happiness["year"].max())

1990 2023
2011 2025


In [5]:
gdi.columns.tolist()

['countryIsoCode',
 'country',
 'indexCode',
 'index',
 'dimension',
 'indicatorCode',
 'indicator',
 'year',
 'yearStr',
 'actualValue',
 'value',
 'note']

In [7]:
# Во всех файлах выбираем общий временной период/We select a common time period across all files/Wir wählen einen gemeinsamen Zeitraum für alle Dateien aus.

hdi_clean = hdi[hdi["year"].between(2011, 2023)].copy()

happiness_clean = happiness[
    happiness["year"].between(2011, 2023)
].copy()

gdi_clean = gdi[
    gdi["year"].between(2011, 2023)
].copy()

In [8]:
print("HDI:", hdi_clean["year"].min(), "-", hdi_clean["year"].max())
print("Happiness:", happiness_clean["year"].min(), "-", happiness_clean["year"].max())
print("GDI:", gdi_clean["year"].min(), "-", gdi_clean["year"].max())

HDI: 2011 - 2023
Happiness: 2011 - 2023
GDI: 2011 - 2023


In [9]:
print(hdi.shape, "→", hdi_clean.shape)
print(happiness.shape, "→", happiness_clean.shape)
print(gdi.shape, "→", gdi_clean.shape)

(6630, 9) → (2535, 9)
(2116, 13) → (1822, 13)
(66537, 12) → (28469, 12)


In [10]:
happiness_clean["country"].nunique()

168

In [11]:
hdi_clean["country"].nunique()

195

In [12]:
gdi_clean["country"].nunique()

206

In [13]:
hdi_clean.groupby("country")["year"].nunique().value_counts()

year
13    195
Name: count, dtype: int64

In [14]:
gdi_clean.loc[
    gdi_clean["countryIsoCode"].str.len() != 3,
    ["countryIsoCode", "country"]
].drop_duplicates()

,countryIsoCode,country
62764,ZZA.VHHD,Very High Human Development
63138,ZZB.HHD,High Human Development
63507,ZZC.MHD,Medium Human Development
63856,ZZD.LHD,Low Human Development
64155,ZZE.AS,Arab States
64504,ZZF.EAP,East Asia And The Pacific
64848,ZZG.ECA,Europe And Central Asia
65167,ZZH.LAC,Latin America And The Caribbean
65501,ZZI.SA,South Asia
65875,ZZJ.SSA,Sub-Saharan Africa


In [15]:
# избавимся от агрегатов в GDI/ We remove the aggregates in the GDI dataset/Wir entfernen die Aggregate aus dem GDI‑Datensatz.
gdi_clean = gdi_clean[
    gdi_clean["countryIsoCode"].str.len() == 3
].copy()

In [16]:
print(gdi_clean["country"].nunique())
print(gdi_clean["countryIsoCode"].nunique())

195
195


In [17]:
#создадим master list / we create a master list/ Wir erstellen eine Master Liste
countries = hdi_clean[
    ["iso3", "country", "region"]
].drop_duplicates().copy()

In [35]:
countries

,iso3,country,region,iso2
0,AFG,Afghanistan,South Asia,AF
1,ALB,Albania,Central and Eastern Europe,AL
2,DZA,Algeria,Middle East and Northern Africa,DZ
3,AND,Andorra,Western Europe,AD
4,AGO,Angola,Sub-Saharan Africa,AO
...,...,...,...,...
190,VEN,Venezuela (Bolivarian Republic of),Latin America and Caribbean,VE
191,VNM,Viet Nam,Eastern Asia,VN
192,YEM,Yemen,Middle East and Northern Africa,YE
193,ZMB,Zambia,Sub-Saharan Africa,ZM


In [19]:
print(iso.shape)
print(iso.columns.tolist())
iso.head(10)

(191, 2)
['country', 'iso_alpha_2']


,country,iso_alpha_2
0,Afghanistan,AF
1,Albania,AL
2,Algeria,DZ
3,Andorra,AD
4,Angola,AO
5,Antigua and Barbuda,AG
6,Argentina,AR
7,Armenia,AM
8,Australia,AU
9,Austria,AT


In [21]:
#Сравним списки стран/ We compare the country lists/ Wir vwergleichen die Länderliste
hdi_country_names = set(countries["country"])
iso_country_names = set(iso["country"])

In [22]:
print("Есть в HDI, но нет в ISO:")
print(hdi_country_names - iso_country_names)

print("\nЕсть в ISO, но нет в HDI:")
print(iso_country_names - hdi_country_names)

Есть в HDI, но нет в ISO:
{'Tanzania (United Republic of)', "Lao People's Democratic Republic", "Côte d'Ivoire", 'Hong Kong, China (SAR)', 'Palestine, State of', 'Congo (Democratic Republic of the)', 'Micronesia (Federated States of)', 'Türkiye', 'Eswatini (Kingdom of)', 'Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Venezuela (Bolivarian Republic of)', "Korea (Democratic People's Rep. of)", 'Moldova (Republic of)'}

Есть в ISO, но нет в HDI:
{'Iran', 'Bolivia', 'Eswatini', 'Taiwan', 'Moldova', 'Venezuela', 'Lao PDR', 'Tanzania', 'Turkey', 'Micronesia'}


In [23]:
countries[
    countries["country"].isin(hdi_country_names - iso_country_names)
].sort_values("country")

,iso3,country,region
4115,BOL,Bolivia (Plurinational State of),Latin America and Caribbean
4134,COD,Congo (Democratic Republic of the),Sub-Saharan Africa
4140,CIV,Côte d'Ivoire,Sub-Saharan Africa
4151,SWZ,Eswatini (Kingdom of),Sub-Saharan Africa
4169,HKG,"Hong Kong, China (SAR)",Eastern Asia
4174,IRN,Iran (Islamic Republic of),South Asia
4185,PRK,Korea (Democratic People's Rep. of),Eastern Asia
4189,LAO,Lao People's Democratic Republic,Eastern Asia
4208,FSM,Micronesia (Federated States of),Eastern Asia
4209,MDA,Moldova (Republic of),Central and Eastern Europe


In [24]:
iso = iso.rename(columns={"iso_alpha_2": "iso2"})

In [25]:
iso.columns

Index(['country', 'iso2'], dtype='object')

In [26]:
countries = countries.merge(
    iso,
    on="country",
    how="left"
)

In [27]:
print(countries.shape)
print(countries["iso2"].isna().sum())

(195, 4)
15


In [28]:
countries[
    countries["iso2"].isna()
].sort_values("country")

,iso3,country,region,iso2
20,BOL,Bolivia (Plurinational State of),Latin America and Caribbean,NaN
39,COD,Congo (Democratic Republic of the),Sub-Saharan Africa,NaN
45,CIV,Côte d'Ivoire,Sub-Saharan Africa,NaN
56,SWZ,Eswatini (Kingdom of),Sub-Saharan Africa,NaN
74,HKG,"Hong Kong, China (SAR)",Eastern Asia,NaN
79,IRN,Iran (Islamic Republic of),South Asia,NaN
90,PRK,Korea (Democratic People's Rep. of),Eastern Asia,NaN
94,LAO,Lao People's Democratic Republic,Eastern Asia,NaN
113,FSM,Micronesia (Federated States of),Eastern Asia,NaN
114,MDA,Moldova (Republic of),Central and Eastern Europe,NaN


In [29]:
iso2_missing = {
    "BOL": "BO",
    "COD": "CD",
    "CIV": "CI",
    "SWZ": "SZ",
    "HKG": "HK",
    "IRN": "IR",
    "PRK": "KP",
    "LAO": "LA",
    "FSM": "FM",
    "MDA": "MD",
    "NAM": "NA",
    "PSE": "PS",
    "TZA": "TZ",
    "TUR": "TR",
    "VEN": "VE"
}

In [30]:
countries["iso2"] = countries["iso2"].fillna(
    countries["iso3"].map(iso2_missing)
)

In [31]:
print(countries.shape)
print(countries["iso2"].isna().sum())
print(countries["iso2"].nunique())

(195, 4)
0
195


In [33]:
#проверка на дубликаты/ we prove duplicates/ wir prüfen, ob es die Duplikate gibt
countries[countries["iso2"].duplicated(keep=False)]

,iso3,country,region,iso2


In [38]:
countries.sort_values("country").head()

,iso3,country,region,iso2
0,AFG,Afghanistan,South Asia,AF
1,ALB,Albania,Central and Eastern Europe,AL
2,DZA,Algeria,Middle East and Northern Africa,DZ
3,AND,Andorra,Western Europe,AD
4,AGO,Angola,Sub-Saharan Africa,AO


In [39]:
# Работаем с GDI/ Now we are working with GDI File/ Jetzt arbeiten wir mit GDI-Datei
master_iso3 = set(countries["iso3"])
gdi_iso3 = set(gdi_clean["countryIsoCode"])
print("Есть в Master, но нет в GDI:")
print(master_iso3 - gdi_iso3)

print("\nЕсть в GDI, но нет в Master:")
print(gdi_iso3 - master_iso3)

Есть в Master, но нет в GDI:
set()

Есть в GDI, но нет в Master:
set()


In [41]:
gdi_countries = gdi_clean[
    ["countryIsoCode", "country"]
].drop_duplicates().copy()

gdi_countries.shape

(195, 2)

In [43]:
country_check = countries.merge(
    gdi_countries,
    left_on="iso3",
    right_on="countryIsoCode",
    how="left",
    suffixes=("_master", "_gdi")
)

In [44]:
country_check[
    country_check["country_master"] != country_check["country_gdi"]
][
    ["iso3", "country_master", "country_gdi"]
]

,iso3,country_master,country_gdi


In [46]:
# Работаем с Happiness/ Now we are working with Happiness File/ Jetzt arbeiten wir mit Happiness-Datei

happiness_country_names = set(happiness_clean["country"])
master_country_names = set(countries["country"])

print("Есть в Master, но нет в Happiness:")
print(master_country_names - happiness_country_names)

print("\nЕсть в Happiness, но нет в Master:")
print(happiness_country_names - master_country_names)

Есть в Master, но нет в Happiness:
{'Tuvalu', 'Cabo Verde', "Lao People's Democratic Republic", "Côte d'Ivoire", 'Hong Kong, China (SAR)', 'Congo (Democratic Republic of the)', 'Timor-Leste', 'Vanuatu', 'Kiribati', 'Marshall Islands', 'Syrian Arab Republic', 'Korea (Republic of)', 'Papua New Guinea', 'Sao Tome and Principe', 'Palau', 'Brunei Darussalam', 'Bahamas', 'Eswatini (Kingdom of)', 'Nauru', 'Iran (Islamic Republic of)', 'Samoa', 'Saint Vincent and the Grenadines', 'Guinea-Bissau', 'Liechtenstein', 'Saint Lucia', 'Micronesia (Federated States of)', 'Eritrea', 'San Marino', 'Bolivia (Plurinational State of)', 'Antigua and Barbuda', 'Barbados', 'Venezuela (Bolivarian Republic of)', 'Andorra', 'Grenada', 'Dominica', 'Saint Kitts and Nevis', 'Tanzania (United Republic of)', 'Solomon Islands', 'Palestine, State of', 'Tonga', 'Fiji', 'Seychelles', "Korea (Democratic People's Rep. of)", 'Moldova (Republic of)', 'Monaco', 'Equatorial Guinea'}

Есть в Happiness, но нет в Master:
{'Republ

In [49]:
happiness_name_map = {
    "Republic of Korea": "Korea (Republic of)",
    "Iran": "Iran (Islamic Republic of)",
    "Côte d’Ivoire": "Côte d'Ivoire",
    "Bolivia": "Bolivia (Plurinational State of)",
    "Eswatini": "Eswatini (Kingdom of)",
    "Swaziland": "Eswatini (Kingdom of)",
    "Hong Kong SAR of China": "Hong Kong, China (SAR)",
    "DR Congo": "Congo (Democratic Republic of the)",
    "Tanzania": "Tanzania (United Republic of)",
    "Venezuela": "Venezuela (Bolivarian Republic of)",
    "Lao PDR": "Lao People's Democratic Republic",
    "Republic of Moldova": "Moldova (Republic of)",
    "State of Palestine": "Palestine, State of",
    "Syria": "Syrian Arab Republic"
}

In [50]:
happiness_clean["country"] = happiness_clean["country"].replace(
    happiness_name_map
)

In [51]:
happiness_country_names = set(happiness_clean["country"])
master_country_names = set(countries["country"])

print("Есть в Master, но нет в Happiness:")
print(master_country_names - happiness_country_names)

print("\nЕсть в Happiness, но нет в Master:")
print(happiness_country_names - master_country_names)

Есть в Master, но нет в Happiness:
{'Tuvalu', 'Cabo Verde', 'Timor-Leste', 'Vanuatu', 'Kiribati', 'Marshall Islands', 'Papua New Guinea', 'Sao Tome and Principe', 'Palau', 'Brunei Darussalam', 'Bahamas', 'Nauru', 'Samoa', 'Saint Vincent and the Grenadines', 'Guinea-Bissau', 'Liechtenstein', 'Saint Lucia', 'Micronesia (Federated States of)', 'Eritrea', 'San Marino', 'Antigua and Barbuda', 'Barbados', 'Andorra', 'Dominica', 'Saint Kitts and Nevis', 'Solomon Islands', 'Tonga', 'Fiji', 'Seychelles', "Korea (Democratic People's Rep. of)", 'Grenada', 'Monaco', 'Equatorial Guinea'}

Есть в Happiness, но нет в Master:
{'North Cyprus', 'Taiwan Province of China', 'Kosovo', 'Somaliland Region', 'Puerto Rico'}


In [52]:
print("Строк:", happiness_clean.shape[0])
print("Уникальных стран:", happiness_clean["country"].nunique())

Строк: 1822
Уникальных стран: 167


In [53]:
print(happiness_clean.columns.tolist())
happiness_clean.head()

['year', 'rank_in_year', 'country', 'happiness_score', 'lower_whisker', 'upper_whisker', 'explained_log_gdp_per_capita', 'explained_social_support', 'explained_healthy_life_expectancy', 'explained_freedom', 'explained_generosity', 'explained_corruption', 'dystopia_plus_residual']


,year,rank_in_year,country,happiness_score,lower_whisker,upper_whisker,explained_log_gdp_per_capita,explained_social_support,explained_healthy_life_expectancy,explained_freedom,explained_generosity,explained_corruption,dystopia_plus_residual
0,2011,1,Denmark,7.856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011,2,Finland,7.579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011,3,Norway,7.524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011,4,Netherlands,7.512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,5,Canada,7.499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
print("Пропусков в rank:", happiness_clean["rank_in_year"].isna().sum())

duplicate_ranks = happiness_clean[
    happiness_clean.duplicated(
        subset=["year", "rank_in_year"],
        keep=False
    )
]

print("Строк с повторяющимися рангами:", duplicate_ranks.shape[0])

Пропусков в rank: 0
Строк с повторяющимися рангами: 2


In [55]:
duplicate_ranks[
    ["year", "rank_in_year", "country", "happiness_score"]
].sort_values(["year", "rank_in_year"])

,year,rank_in_year,country,happiness_score
1057,2018,120,Gambia,4.516
1058,2018,120,Sri Lanka,4.366


In [56]:
happiness_clean[
    (happiness_clean["year"] == 2018) &
    (happiness_clean["rank_in_year"].between(115, 125))
][
    ["year", "rank_in_year", "country", "happiness_score"]
].sort_values("rank_in_year")

,year,rank_in_year,country,happiness_score
1052,2018,115,Burkina Faso,4.587
1053,2018,116,Armenia,4.559
1054,2018,117,Iran (Islamic Republic of),4.548
1055,2018,118,Guinea,4.534
1056,2018,119,Georgia,4.519
1057,2018,120,Gambia,4.516
1058,2018,120,Sri Lanka,4.366
1059,2018,121,Kenya,4.509
1060,2018,122,Mauritania,4.490
1061,2018,123,Mozambique,4.466


In [57]:
happiness_clean[
    (happiness_clean["year"] == 2018) &
    (happiness_clean["happiness_score"].between(4.30, 4.40))
][
    ["year", "rank_in_year", "country", "happiness_score"]
].sort_values("happiness_score", ascending=False)

,year,rank_in_year,country,happiness_score
1066,2018,128,Mali,4.390
1067,2018,129,Sierra Leone,4.374
1058,2018,120,Sri Lanka,4.366
1068,2018,131,Myanmar,4.360
1069,2018,132,Chad,4.350
1070,2018,133,Ukraine,4.332


In [58]:
happiness_clean.loc[
    (happiness_clean["year"] == 2018) &
    (happiness_clean["country"] == "Sri Lanka"),
    "rank_in_year"
] = 130

In [59]:
print("Пропусков в rank:", happiness_clean["rank_in_year"].isna().sum())

duplicate_ranks = happiness_clean[
    happiness_clean.duplicated(
        subset=["year", "rank_in_year"],
        keep=False
    )
]

print("Строк с повторяющимися рангами:", duplicate_ranks.shape[0])

Пропусков в rank: 0
Строк с повторяющимися рангами: 0


In [60]:
# сливаем с мастер-листом
happiness_clean = happiness_clean.merge(
    countries[["country", "iso3", "iso2", "region"]],
    on="country",
    how="left",
    validate="many_to_one"
)

In [61]:
print(happiness_clean.shape)

print(
    happiness_clean[
        happiness_clean["iso3"].isna()
    ]["country"].unique()
)

(1822, 16)
['Puerto Rico' 'Taiwan Province of China' 'Kosovo' 'Somaliland Region'
 'North Cyprus']


In [62]:
happiness_only = happiness_clean[
    happiness_clean["iso3"].isna()
]

happiness_only.groupby("country").agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    rows=("year", "count")
)

,first_year,last_year,rows
country,,,
Kosovo,2011,2023,12
North Cyprus,2012,2021,9
Puerto Rico,2011,2015,2
Somaliland Region,2011,2015,4
Taiwan Province of China,2011,2023,12


In [63]:
happiness_clean.loc[
    happiness_clean["country"] == "Puerto Rico",
    ["iso3", "iso2"]
] = ["PRI", "PR"]

happiness_clean.loc[
    happiness_clean["country"] == "Taiwan Province of China",
    ["iso3", "iso2"]
] = ["TWN", "TW"]

happiness_clean.loc[
    happiness_clean["country"] == "Kosovo",
    ["iso3", "iso2"]
] = ["XKX", "XK"]

In [64]:
happiness_clean.loc[
    happiness_clean["iso2"].isna(),
    ["country", "iso3", "iso2"]
].drop_duplicates()

,country,iso3,iso2
110,Somaliland Region,NaN,NaN
224,North Cyprus,NaN,NaN


In [65]:
happiness_final = happiness_clean[
    [
        "year",
        "country",
        "iso3",
        "iso2",
        "region",
        "happiness_score",
        "rank_in_year"
    ]
].copy()

In [66]:
print("Размер:", happiness_final.shape)
print("Стран:", happiness_final["country"].nunique())
print("Дубликатов country-year:",
      happiness_final.duplicated(["country", "year"]).sum())

Размер: (1822, 7)
Стран: 167
Дубликатов country-year: 0


In [67]:
happiness_final.loc[
    (happiness_final["country"] == "Sri Lanka") &
    (happiness_final["year"] == 2018)
]

,year,country,iso3,iso2,region,happiness_score,rank_in_year
1058,2018,Sri Lanka,LKA,LK,South Asia,4.366,130


In [68]:
#СОХРАНЯЕМ/ SAVE /SPEICHERN
happiness_final.to_csv(
    "happiness_clean_2011_2023.csv",
    index=False
)

In [69]:
happiness_final.to_excel(
    "happiness_clean_2011_2023.xlsx",
    index=False
)

In [79]:
# HDI 
print(hdi_clean.columns.tolist())

['iso3', 'country', 'region', 'year', 'hdi', 'life_expectancy', 'education_expected', 'education_mean', 'income', 'iso2']


In [80]:
hdi_clean = hdi_clean.merge(
    countries[["iso3", "iso2"]],
    on="iso3",
    how="left",
    validate="many_to_one"
)

In [83]:
hdi_clean = hdi_clean.drop(columns=["iso2_y"])

hdi_clean = hdi_clean.rename(
    columns={"iso2_x": "iso2"}
)

In [86]:
print("Rows:", len(hdi_clean))
print("Columns:", hdi_clean.columns.tolist())
print("Missing iso2:", hdi_clean["iso2"].isna().sum())
print("Duplicate country-year:",
      hdi_clean.duplicated(["iso3", "year"]).sum())

Rows: 2535
Columns: ['iso3', 'country', 'region', 'year', 'hdi', 'life_expectancy', 'education_expected', 'education_mean', 'income', 'iso2']
Missing iso2: 0
Duplicate country-year: 0


In [85]:
#СОХРАНЯЕМ/ SAVE /SPEICHERN
hdi_clean.to_csv(
    "hdi_clean_2011_2023.csv",
    index=False
)
hdi_clean.to_excel(
    "hdi_clean_2011_2023.xlsx",
    index=False
)

In [72]:
print(hdi_clean.shape)
print("Пропусков iso2:", hdi_clean["iso2"].isna().sum())
print("Дубликатов country-year:",
      hdi_clean.duplicated(["iso3", "year"]).sum())

(2535, 10)
Пропусков iso2: 0
Дубликатов country-year: 0


In [87]:
gdi_clean["indicator"].drop_duplicates().tolist()

['Expected Years of Schooling, female (years)',
 'Expected Years of Schooling, male (years)',
 'Gender Development Index (value)',
 'GDI Group',
 'Gross National Income Per Capita, female (2021 PPP$)',
 'Gross National Income Per Capita, male (2021 PPP$)',
 'HDI female',
 'HDI male',
 'Life Expectancy at Birth, female (years)',
 'Life Expectancy at Birth, male (years)',
 'Mean Years of Schooling, female (years)',
 'Mean Years of Schooling, male (years)']

In [88]:
gdi_indicators = [
    "Expected Years of Schooling, female (years)",
    "Expected Years of Schooling, male (years)",
    "Gross National Income Per Capita, female (2021 PPP$)",
    "Gross National Income Per Capita, male (2021 PPP$)",
    "Life Expectancy at Birth, female (years)",
    "Life Expectancy at Birth, male (years)",
    "Mean Years of Schooling, female (years)",
    "Mean Years of Schooling, male (years)"
]

In [89]:
gdi_selected = gdi_clean[
    gdi_clean["indicator"].isin(gdi_indicators)
].copy()

In [90]:
print("Размер:", gdi_selected.shape)

print("\nИндикаторов:",
      gdi_selected["indicator"].nunique())

print("\nСписок:")
print(gdi_selected["indicator"].value_counts())

Размер: (19734, 12)

Индикаторов: 8

Список:
indicator
Life Expectancy at Birth, male (years)                  2535
Life Expectancy at Birth, female (years)                2535
Expected Years of Schooling, female (years)             2520
Expected Years of Schooling, male (years)               2520
Mean Years of Schooling, female (years)                 2481
Mean Years of Schooling, male (years)                   2481
Gross National Income Per Capita, male (2021 PPP$)      2331
Gross National Income Per Capita, female (2021 PPP$)    2331
Name: count, dtype: int64


In [91]:
duplicates = gdi_selected.duplicated(
    subset=["countryIsoCode", "year", "indicator"],
    keep=False
)

print("Строк-дубликатов:", duplicates.sum())

Строк-дубликатов: 0


In [92]:
gdi_selected[
    [
        "countryIsoCode",
        "country",
        "year",
        "indicator",
        "actualValue",
        "value"
    ]
].head(12)

,countryIsoCode,country,year,indicator,actualValue,value
21,AFG,Afghanistan,2011,"Expected Years of Schooling, female (years)",7.385710,7.386
22,AFG,Afghanistan,2012,"Expected Years of Schooling, female (years)",7.606157,7.606
23,AFG,Afghanistan,2013,"Expected Years of Schooling, female (years)",7.826604,7.827
24,AFG,Afghanistan,2014,"Expected Years of Schooling, female (years)",8.047050,8.047
25,AFG,Afghanistan,2015,"Expected Years of Schooling, female (years)",8.032045,8.032
26,AFG,Afghanistan,2016,"Expected Years of Schooling, female (years)",8.017040,8.017
27,AFG,Afghanistan,2017,"Expected Years of Schooling, female (years)",8.002035,8.002
28,AFG,Afghanistan,2018,"Expected Years of Schooling, female (years)",7.987030,7.987
29,AFG,Afghanistan,2019,"Expected Years of Schooling, female (years)",8.020075,8.020
30,AFG,Afghanistan,2020,"Expected Years of Schooling, female (years)",8.053257,8.053


In [93]:
indicator_names = {
    "Life Expectancy at Birth, female (years)": "life_expectancy_female",
    "Life Expectancy at Birth, male (years)": "life_expectancy_male",
    
    "Expected Years of Schooling, female (years)": "education_expected_female",
    "Expected Years of Schooling, male (years)": "education_expected_male",
    
    "Mean Years of Schooling, female (years)": "education_mean_female",
    "Mean Years of Schooling, male (years)": "education_mean_male",
    
    "Gross National Income Per Capita, female (2021 PPP$)": "income_female",
    "Gross National Income Per Capita, male (2021 PPP$)": "income_male"
}

In [94]:
gdi_selected["indicator"] = gdi_selected["indicator"].replace(
    indicator_names
)

In [95]:
print(gdi_selected["indicator"].unique())

['education_expected_female' 'education_expected_male' 'income_female'
 'income_male' 'life_expectancy_female' 'life_expectancy_male'
 'education_mean_female' 'education_mean_male']


In [96]:
gdi_selected.head()

,countryIsoCode,country,indexCode,index,dimension,indicatorCode,indicator,year,yearStr,actualValue,value,note
21,AFG,Afghanistan,GDI,Gender Development Index,NaN,eys_f,education_expected_female,2011,2011,7.385710,7.386,NaN
22,AFG,Afghanistan,GDI,Gender Development Index,NaN,eys_f,education_expected_female,2012,2012,7.606157,7.606,NaN
23,AFG,Afghanistan,GDI,Gender Development Index,NaN,eys_f,education_expected_female,2013,2013,7.826604,7.827,NaN
24,AFG,Afghanistan,GDI,Gender Development Index,NaN,eys_f,education_expected_female,2014,2014,8.047050,8.047,NaN
25,AFG,Afghanistan,GDI,Gender Development Index,NaN,eys_f,education_expected_female,2015,2015,8.032045,8.032,NaN


In [97]:
extra_indicators = [
    "Gender Development Index (value)",
    "GDI Group",
    "HDI female",
    "HDI male"
]

gdi_clean[
    gdi_clean["indicator"].isin(extra_indicators)
].groupby("indicator").agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    rows=("year", "count"),
    countries=("countryIsoCode", "nunique")
)

,first_year,last_year,rows,countries
indicator,,,,
GDI Group,2023,2023,184,184
Gender Development Index (value),2011,2023,2326,184
HDI female,2011,2023,2326,184
HDI male,2011,2023,2326,184


In [98]:
all_gdi_indicators = gdi_indicators + extra_indicators

In [99]:
gdi_selected = gdi_clean[
    gdi_clean["indicator"].isin(all_gdi_indicators)
].copy()

In [100]:
indicator_names.update({
    "Gender Development Index (value)": "gdi",
    "GDI Group": "gdi_group",
    "HDI female": "hdi_female",
    "HDI male": "hdi_male"
})

In [101]:
gdi_selected["indicator"] = gdi_selected["indicator"].replace(
    indicator_names
)

In [102]:
print("Строк:", gdi_selected.shape[0])
print("Индикаторов:", gdi_selected["indicator"].nunique())

print(gdi_selected["indicator"].value_counts())

Строк: 26896
Индикаторов: 12
indicator
life_expectancy_female       2535
life_expectancy_male         2535
education_expected_female    2520
education_expected_male      2520
education_mean_male          2481
education_mean_female        2481
income_male                  2331
income_female                2331
hdi_male                     2326
gdi                          2326
hdi_female                   2326
gdi_group                     184
Name: count, dtype: int64


In [103]:
#Делаем пивотирование/ we are making pivot
gdi_wide = gdi_selected.pivot(
    index=["countryIsoCode", "country", "year"],
    columns="indicator",
    values="actualValue"
).reset_index()


In [104]:
print(gdi_wide.shape)
print(gdi_wide.columns.tolist())

gdi_wide.head()

(2535, 15)
['countryIsoCode', 'country', 'year', 'education_expected_female', 'education_expected_male', 'education_mean_female', 'education_mean_male', 'gdi', 'gdi_group', 'hdi_female', 'hdi_male', 'income_female', 'income_male', 'life_expectancy_female', 'life_expectancy_male']


indicator,countryIsoCode,country,year,education_expected_female,education_expected_male,education_mean_female,education_mean_male,gdi,gdi_group,hdi_female,hdi_male,income_female,income_male,life_expectancy_female,life_expectancy_male
0,AFG,Afghanistan,2011,7.385710,11.460760,0.595375,3.13969,0.685,NaN,0.371719,0.542445,1140.727037,4719.872325,62.852,59.625
1,AFG,Afghanistan,2012,7.606157,11.896333,0.633786,3.21379,0.653,NaN,0.363717,0.556699,885.649014,5366.293623,63.344,60.099
2,AFG,Afghanistan,2013,7.826604,12.331907,0.672197,3.28789,0.681,NaN,0.384286,0.563890,1176.053698,5273.129660,63.814,60.534
3,AFG,Afghanistan,2014,8.047050,12.767480,0.710609,3.36199,0.673,NaN,0.383857,0.570062,1071.594603,5311.945717,64.009,60.495
4,AFG,Afghanistan,2015,8.032045,12.816715,0.749020,3.43609,0.695,NaN,0.394032,0.566751,1275.712382,4860.966850,64.178,60.363


In [105]:
gdi_wide = gdi_wide.rename(
    columns={"countryIsoCode": "iso3"}
)

In [106]:
gdi_wide.columns.tolist()

['iso3',
 'country',
 'year',
 'education_expected_female',
 'education_expected_male',
 'education_mean_female',
 'education_mean_male',
 'gdi',
 'gdi_group',
 'hdi_female',
 'hdi_male',
 'income_female',
 'income_male',
 'life_expectancy_female',
 'life_expectancy_male']

In [107]:
gdi_final = gdi_wide.merge(
    countries[["iso3", "iso2", "region"]],
    on="iso3",
    how="left",
    validate="many_to_one"
)

In [108]:
print("Размер:", gdi_final.shape)

print("Стран:", gdi_final["iso3"].nunique())

print("Пропусков iso2:",
      gdi_final["iso2"].isna().sum())

print("Пропусков region:",
      gdi_final["region"].isna().sum())

print("Дубликатов country-year:",
      gdi_final.duplicated(["iso3", "year"]).sum())

Размер: (2535, 17)
Стран: 195
Пропусков iso2: 0
Пропусков region: 0
Дубликатов country-year: 0


In [109]:
gdi_final = gdi_final[
    [
        "iso3",
        "iso2",
        "country",
        "region",
        "year",
        "gdi",
        "gdi_group",
        "hdi_female",
        "hdi_male",
        "life_expectancy_female",
        "life_expectancy_male",
        "education_expected_female",
        "education_expected_male",
        "education_mean_female",
        "education_mean_male",
        "income_female",
        "income_male"
    ]
].copy()

In [110]:
print("Размер:", gdi_final.shape)
print("Стран:", gdi_final["iso3"].nunique())
print("Годы:", gdi_final["year"].min(), "-", gdi_final["year"].max())

print(
    "Дубликатов country-year:",
    gdi_final.duplicated(["iso3", "year"]).sum()
)

print("\nПропуски:")
print(gdi_final.isna().sum())

Размер: (2535, 17)
Стран: 195
Годы: 2011 - 2023
Дубликатов country-year: 0

Пропуски:
iso3                            0
iso2                            0
country                         0
region                          0
year                            0
gdi                           209
gdi_group                    2351
hdi_female                    209
hdi_male                      209
life_expectancy_female          0
life_expectancy_male            0
education_expected_female      15
education_expected_male        15
education_mean_female          54
education_mean_male            54
income_female                 204
income_male                   204
dtype: int64


In [111]:
gdi_final.to_csv(
    "GDI_clean_2011_2023.csv",
    index=False
)

gdi_final.to_excel(
    "GDI_clean_2011_2023.xlsx",
    index=False
)